## Corrigida a referência de eNB entre MeContext e ManagedElement

In [ ]:
from datetime import datetime
semana=datetime.today().strftime('%U')
semana_atual= int(semana) + 1
print(semana_atual)

In [ ]:
# Função para carregar os arquivos
base_path = r"C:\Dumps\todos_enms"

def load_parquet(pattern, columns=None):
    import glob, pandas as pd
    files = glob.glob(pattern)
    if not files:
        return pd.DataFrame()
    df = pd.concat((pd.read_parquet(f, columns=columns) for f in files),
                   ignore_index=True)
    try:
        df = df.convert_dtypes(dtype_backend="pyarrow")
    except Exception:
        pass
    return df


In [ ]:
# TermPointToMme

import pandas as pd

df1= pd.read_csv('C:/Dumps/LTE_Total/TermPointToMme.txt', sep='\t', usecols=['UF', 'eNB', 'Termpoint','administrativeState','ipAddress1','ipAddress2'])

science= pd.read_excel('C:/Camilo/SCIENCE.xlsx')

df2= pd.merge(df1,science, left_on='eNB', right_on= 'SITE', how= 'left')

df3= df2.loc[:, ['UF', 'MUNICÍPIO', 'BAIRRO','eNB', 'Termpoint', 'administrativeState','ipAddress1','ipAddress2']]

#df2.loc[:, ['UF', 'MUNICÍPIO', 'BAIRRO','eNB', 'Termpoint','administrativeState','operationalState','ipAddress1','ipAddress2']]

df4= df3.set_index('UF')

df4.drop_duplicates(subset=['eNB', 'Termpoint', 'administrativeState'], inplace= True)

arquivo_txt= 'C:/Camilo/' + 'TermpointToMme'+ '_'+str(semana_atual)+'.txt'
arquivo_csv= 'C:/Vivo Reports/LTE/LTE MOs/' + 'TermpointToMme'+'.csv'

df4.to_csv(arquivo_txt, sep='\t', index=None)
# df4.to_csv(arquivo_csv, encoding='ISO-8859-1', index=None)

print('Finished')

In [ ]:
# EUtranCellFDD

df1= pd.read_csv('C:/Dumps/LTE_total/EUtranCellFDD.txt', sep='\t', usecols=['UF','eNB','CELL','cellId', 'earfcndl', 'dlChannelBandwidth', 
                 'tac', 'physicalLayerCellIdGroup', 'physicalLayerSubCellId','rachRootSequence','catm1SupportEnabled','PCI','crsGain', 'primaryPlmnReserved', 'administrativeState', 
                 'operationalState','qRxLevMin','qQualMin', 'cellRange','primaryUpperLayerInd'],
                 low_memory=False)

df1['Banda']= df1['CELL'].str[0:1].replace(['T','Q', 'V', 'Z', 'U', 'P', 'Y', 'C'],
                                           ['2600','2600', '1800', '700', '2100', '2600', '850', '1800'])
# Com o comando acima o eNB viria com "ManagedElement="" antes do nome da eNB, para deixar apenas o eNB name, usar o comando abaixo
df1= df1.copy()
# df1['eNB']=df1['eNB'].str.replace('ManagedElement=',"")
#df1['UF']= df1['eNB'].str[-2:].isin(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE']) # selecionar as UFs desejadas

df1['UF']= df1['eNB'].str[-2:]
df1['PCI']= df1['physicalLayerCellIdGroup']*3 + df1['physicalLayerSubCellId']

df1.dropna(subset='cellId',inplace=True)

# df1.drop_duplicates(subset=['eNB', 'CELL'],inplace=True)

science= pd.read_excel('C:/Camilo/cellref.xlsx', usecols=['CN','CLUSTER','BAIRRO', 'ENDERECO','eNB', 'CELL', 'Longitude',
                                                          'Latitude', 'Azimuth', 'Height', 'SiteType', 'Site_Name'])

science.rename(columns={'CLUSTER': 'MUNICIPIO', 'Azimuth':'Azimute', 'Height': 'Altura'}, inplace=True)

#science1= science.loc[:, ['SITE', 'MUNICÍPIO', 'BAIRRO']]

# Cascata de DOIS niveis para trazer o dado fisico do cellref.
#
# NIVEL 1 - por CELULA. E a unica chave que atravessa rede e cadastro: num Hotel
# de BTS o no hospeda sites remotos, entao a chave pelo no (UF + eNB[1:4]) junta
# hospede e hospedeiro no mesmo balde e o drop_duplicates apaga um deles.
# Caso medido (MCRD3_AM, Manaus): 18 linhas corretas do cellref viravam 3, e a
# coordenada da UFAM 6 era carimbada em TCRD11/UCRD21/VCRD21/ZCRD21, a 2,7 km
# dali. Efeito nacional do bug: 230 celulas erradas em 30 nos.
#
# NIVEL 2 - reserva pela chave do no + setor, o comportamento antigo. Serve para
# celula que o cadastro ainda nao tem (site novo: o science demora a atualizar).
#
# A coluna Origem_coord diz de qual nivel veio cada linha, para os relatorios a
# jusante saberem em que coordenada confiar.
fisicas = ['CN', 'MUNICIPIO', 'BAIRRO', 'ENDERECO', 'Site_Name',
           'Longitude', 'Latitude', 'Azimute', 'Altura', 'SiteType']

# --- nivel 1: casamento exato por (eNB, CELL)
science_cel = science.drop_duplicates(subset=['eNB', 'CELL'])
df2 = pd.merge(df1, science_cel, on=['eNB', 'CELL'], how='left')
achou_por_celula = df2['Latitude'].notna()

# --- nivel 2: reserva pela chave do no + setor (comportamento pre-correcao)
science_setor= science.copy()
science_setor['Setor']= science_setor['CELL'].str[-1].replace(['4','5','6', '7', '8', '9'], ['1','2','3','1','2','3'])
science_setor['Site']= science_setor['eNB'].str[-2:] + science_setor['eNB'].str[1:4]
science_setor.drop(columns=['eNB', 'CELL'],inplace=True)
science_setor.drop_duplicates(subset=['Site', 'Setor'],inplace=True)

df2['Site']= df2['eNB'].str[-2:] + df2['eNB'].str[1:4]
df2['Setor']= df2['CELL'].str[-1].replace(['4','5','6', '7', '8', '9'], ['1','2','3','1','2','3'])

# reindexado na ordem de df2 para o preenchimento posicional ser seguro
reserva = df2[['Site', 'Setor']].merge(science_setor, on=['Site', 'Setor'], how='left')
falta = ~achou_por_celula
for coluna in fisicas:
    df2.loc[falta, coluna] = reserva.loc[falta.values, coluna].values

df2['Origem_coord'] = 'sem_cadastro'
df2.loc[achou_por_celula, 'Origem_coord'] = 'celula'
df2.loc[falta & df2['Latitude'].notna(), 'Origem_coord'] = 'herdada_do_no'
print(df2['Origem_coord'].value_counts().to_string())

df3= df2.loc[:, ['UF', 'CN', 'MUNICIPIO', 'BAIRRO','ENDERECO', 'Site_Name','eNB','CELL','cellId', 'Banda', 'earfcndl', 'dlChannelBandwidth', 
                 'tac', 'physicalLayerCellIdGroup', 'physicalLayerSubCellId','rachRootSequence','catm1SupportEnabled','PCI','crsGain', 'primaryPlmnReserved', 'administrativeState', 
                 'operationalState','qRxLevMin','qQualMin', 'cellRange','primaryUpperLayerInd',
                'Latitude', 'Longitude', 'Azimute', 'Altura', 'SiteType', 'Origem_coord']]

df4= df3.set_index('UF')

#pd.set_option('display.max_columns',39)

df4.drop_duplicates(subset=['eNB', 'CELL'], inplace= True)

arquivo_txt= 'C:/Camilo/' + 'EUtranCellFDD'+ '_'+str(semana_atual)+'.txt'
# arquivo_csv= 'C:/Camilo/' + 'EUtranCellFDD'+ '_'+str(semana_atual)+'.csv'

df4.to_csv(arquivo_txt, sep='\t')
# df4.to_csv(arquivo_csv, encoding='ISO-8859-1')

print('Finished')

In [ ]:
# ENodeBFunction

df1= pd.read_csv('C:/Dumps/LTE_total/ENodeBFunction.txt', sep='\t', low_memory=False)



df2= pd.DataFrame(df1)

#df2.drop(['eNodeBFunctionId', 'ENodeBFunctionId'], axis='columns', inplace=True) 

df3= df2.dropna(thresh=1, axis='columns')
#lista= df3.columns.to_list()
#print(lista)

science= pd.read_excel('C:/Camilo/SCIENCE.xlsx')

science= science.loc[:, ['MUNICÍPIO', 'BAIRRO', 'SITE']]

df5= pd.merge(df3,science, left_on='eNB', right_on= 'SITE', how= 'left')
df5.dropna(subset=['alignTtiBundWUlTrigSinr', 'allowMocnCellLevelCommonTac'], inplace=True)

colunas= ['FileName', 'MO', 'UF', 'MUNICÍPIO', 'BAIRRO','eNB', 'alignTtiBundWUlTrigSinr',
 'allowMocnCellLevelCommonTac',
 'altNasBackTo',
 'biasThpWifiMobility',
 'caAwareMfbiIntraCellHo',
 'caUnknownImeisvEnabled',
 'checkEmergencySoftLock',
 'combCellSectorSelectThreshRx',
 'combCellSectorSelectThreshTx',
 'csfbForDualRadioUE',
 'csfbMeasFromIdleMode',
 'csfbUseRegisteredLai',
 'csmMinHighHitThreshold',
 'dlMaxWaitingTimeGlobal',
 'dnsLookupOnTai',
 'dnsLookupTimer',
 'dscpLabel',
 'ductIntCharInfoScheme',
 'enabledUlTrigMeas',
 'eNBId',
 'endcDataUsageReportEnabled',
 'endcPowerOffsetLte',
 'endcS1OverlapMode',
 'endcSplitAllowedMoVoice',
 'endcSplitAllowedNonDynPwrShUe',
 'endcX2IpAddrViaS1Active',
 'eNodeBFunctionId',
 'ENodeBFunctionId',
 'eNodeBPlmnId_mcc',
 'eNodeBPlmnId_mnc',
 'eNodeBPlmnId_mncLength',
 'eranUlCompReSelectLinkTime',
 'eranVlanPortRef',
 'extendedWaitTimeNb',
 'forcedSiTunnelingActive',
 'gtpuErrorIndicationDscp',
 'inactivitySupervisionTimerNb',
 'initPreschedulingEnable',
 'interEnbCaTunnelDscp',
 'interEnbUlCompTunnelDscp',
 'intraEnbHoNasNonDelIndCause',
 'ipAddress',
 'iuaBbmResourceLimit',
 'licCapDistrMethod',
 'licConnectedUsersPercentileConf',
 'licDlBbPercentileConf',
 'licDlPrbPercentileConf',
 'licUlBbPercentileConf',
 'licUlPrbPercentileConf',
 'maxNoCellsNaccCsfb',
 'maxNoCellsNaccSessionCont',
 'maxNrOfEnbPartnersEranUlComp',
 'maxNrOfInterEnbUlCompLbm',
 'maxRandc',
 'measuringEcgiWithAgActive',
 'mfbiFbipOnX2Enabled',
 'mfbiSupport',
 'mfbiSupportPolicy',
 'minRandc',
 'mtRreWithoutNeighborActive',
 'nbRrcConnReestActive',
 'nnsfMode',
 'obsMode',
 'obsProtectionAutoTargetMode',
 'obsStatus',
 'pdbMeasurementDelta',
 'prioritizeAdditionalBands',
 'pwsPersistentStorage',
 'randUpdateInterval',
 'release',
 'releaseInactiveUesInactTime',
 'releaseInactiveUesMpLoadLevel',
 'rrcConnReestActive',
 's1GtpuEchoDscp',
 's1GtpuEchoEnable',
 's1GtpuEchoFailureAction',
 's1GtpuEchoInterval',
 's1HODirDataPathAvail',
 's1RetryTimer',
 'sctpRef',
 'softLockRwRWaitTimerInternal',
 'softLockRwRWaitTimerOperator',
 'srsPeriodicityTdd',
 'tddVoipDrxProfileId',
 'thpWifiMobilityStatus',
 'timeAndPhaseSynchAlignment',
 'timeAndPhaseSynchCritical',
 'timePhaseMaxDeviation',
 'timePhaseMaxDeviationCdma2000',
 'timePhaseMaxDeviationEdrx',
 'timePhaseMaxDeviationIeNbCa',
 'timePhaseMaxDeviationMbms',
 'timePhaseMaxDeviationOtdoa',
 'timePhaseMaxDeviationSib16',
 'timePhaseMaxDeviationTdd',
 'timePhaseMaxDeviationTdd1',
 'timePhaseMaxDeviationTdd2',
 'timePhaseMaxDeviationTdd3',
 'timePhaseMaxDeviationTdd4',
 'timePhaseMaxDeviationTdd5',
 'timePhaseMaxDeviationTdd6',
 'timePhaseMaxDeviationTdd7',
 'timePhaseMaxDevIeNBUlComp',
 'tMoVoiceBearerEstSupervision',
 'tOutgoingHoExecCdma1xRtt',
 'tRelocOverall',
 'tS1HoCancelTimer',
 'tWaitForConnEstInd',
 'ulHeavyUeBsrHyst',
 'ulHeavyUeBsrThresh',
 'ulHeavyUeBsrThUlCaFactor',
 'ulHeavyUeDetectTime',
 'ulHeavyUeDetectTimeOffset',
 'ulMaxWaitingTimeGlobal',
 'ulPhyProcResTradingEnabled',
 'ulSchedulerDynamicBWAllocationEnabled',
 'upIpAddress',
 'upIpAddressRef',
 'useBandPrioritiesInSCellEval',
 'useBandPrioritiesInSib1',
 'useFullResumeId',
 'userLabel',
 'x2BlackList_enbId',
 'x2BlackList_mcc',
 'x2BlackList_mnc',
 'x2BlackList_mncLength',
 'x2GtpuEchoDscp',
 'x2GtpuEchoEnable',
 'x2IpAddrViaS1Active',
 'x2retryTimerMaxAuto',
 'x2retryTimerStart',
 'x2SetupTwoWayRelations',
 'zzzTemporary10',
 'zzzTemporary11',
 'zzzTemporary12',
 'zzzTemporary13',
 'zzzTemporary16',
 'zzzTemporary18',
 'zzzTemporary21',
 'zzzTemporary22',
 'zzzTemporary23',
 'zzzTemporary24',
 'zzzTemporary25',
 'zzzTemporary26',
 'zzzTemporary28',
 'zzzTemporary3',
 'zzzTemporary32',
 'zzzTemporary33',
 'zzzTemporary34',
 'zzzTemporary35',
 'zzzTemporary36',
 'zzzTemporary37',
 'zzzTemporary39',
 'zzzTemporary40',
 'zzzTemporary41',
 'zzzTemporary42',
 'zzzTemporary43',
 'zzzTemporary47',
 'zzzTemporary48',
 'zzzTemporary49',
 'zzzTemporary5',
 'zzzTemporary50',
 'zzzTemporary51',
 'zzzTemporary52',
 'zzzTemporary53',
 'zzzTemporary54',
 'zzzTemporary55',
 'zzzTemporary56',
 'zzzTemporary57',
 'zzzTemporary58',
 'zzzTemporary61',
 'zzzTemporary62',
 'zzzTemporary64',
 'zzzTemporary65',
 'zzzTemporary66',
 'zzzTemporary67',
 'zzzTemporary68',
 'zzzTemporary69',
 'zzzTemporary70',
 'zzzTemporary71',
 'zzzTemporary74',
 'zzzTemporary75',
 'zzzTemporary76',
 'zzzTemporary77',
 'zzzTemporary9']

for coluna in colunas:
    if coluna not in df5.columns:
        df5[coluna] = ''

df6= df5.loc[:, colunas]

df7= df6.set_index('FileName')

df7.drop_duplicates(subset='eNB', inplace= True)

arquivo_txt= 'C:/Camilo/' + 'ENodeBFunction'+ '_'+str(semana_atual)+'.txt'
arquivo_csv= 'C:/Vivo Reports/LTE/LTE MOs/' + 'ENodeBFunction'+ '_'+str(semana_atual)+'.csv'

df7.to_csv(arquivo_txt, sep='\t')
df7.to_csv(arquivo_csv, encoding='ISO-8859-1')

print('Finished')

In [ ]:
import pandas as pd
import numpy as np

arquivo_txt= 'C:/Camilo/' + 'EUtranCellFDD'+ '_'+str(semana_atual)+'.txt'

df1= pd.read_csv('C:/Dumps/LTE_total/ENodeBFunction.txt', delimiter='\t', engine='python')
df= pd.read_csv(arquivo_txt, delimiter='\t', engine='python')

juntar= pd.merge(df, df1, left_on= ['UF', 'eNB'], right_on=['UF', 'eNB'])

# juntar = juntar.apply(lambda col: col.astype(int) if (col.dtype == 'float64' and all(col % 1 == 0)) else col)

juntar['CGI']= juntar['eNodeBPlmnId_mcc'].astype(int).astype(str)+juntar['eNodeBPlmnId_mnc'].astype(int).astype(str)+'-'+juntar['eNBId'].astype(int).astype(str)+\
'-'+juntar['cellId'].astype(int).astype(str)

juntar1= juntar.loc[:, ['UF', 'CN','MUNICIPIO', 'BAIRRO', 'Site_Name','ENDERECO','eNB','eNBId','CELL', 'cellId', 'earfcndl', 'dlChannelBandwidth','tac', 'PCI',
                         'physicalLayerCellIdGroup', 'physicalLayerSubCellId','rachRootSequence','catm1SupportEnabled','CGI','crsGain','cellRange',
                         'primaryPlmnReserved', 'primaryUpperLayerInd','Latitude', 'Longitude','Azimute','Altura','administrativeState', 'operationalState', 'SiteType']]


juntar1['tac']= juntar1['tac'].astype(int)
juntar1['cellId']= juntar1['cellId'].astype(int)

juntar1['SITE']=juntar1['eNB'].str[1:4] + '_' + juntar1['UF']

juntar2= juntar1.set_index('UF')

juntar2.drop_duplicates(subset=['eNB', 'CELL'], inplace= True)

In [ ]:
juntar2.head()

In [ ]:
# Definindo os caminhos dos arquivos de saída (em formatos TXT e CSV)
arquivo_txt = 'C:/Camilo/' + 'EUtranCellFDD' + '_' + str(semana_atual) + '.txt'
# arquivo_csv = 'C:/Camilo/' + 'EUtranCellFDD' + '_' + str(semana_atual) + '.csv'

# Salvando o DataFrame nos arquivos 
juntar2.to_csv('C:/Camilo/EUtranCellFDD.csv',encoding='ISO-8859-1', errors='replace')
juntar2.to_csv(arquivo_txt, sep='\t')
# juntar2.to_csv(arquivo_csv, encoding='ISO-8859-1')

# Imprimindo "Done" no console para indicar que o script foi concluído com sucesso
print('Done')

import pandas as pd
from glob import glob

filenames = glob('C:/Dumps/UMTS All/Export_UtranCell_*.txt')


df1= pd.DataFrame()

for f in filenames:
    df= pd.read_csv(f,delimiter='\t', engine='python')
    df1=pd.concat([df1,df],sort=False, axis=0)

df1['WCELL_NAME']=df1['MO'].str[-6:]
    
df1['WBTS']=df1['iubLinkRef'].str[104:109]
    
df1['RNC_NAME']=df1['MO'].str.extract("(SubNetwork=RNC\w+)",expand=True)
df1['RNC_NAME']=df1['RNC_NAME'].str.replace('SubNetwork=',"")

df1['lac']=df1['locationAreaRef'].str.extract("(LocationArea=\w+)",expand=True)
df1['lac']=df1['lac'].str.replace('LocationArea=',"").astype(int)

df1['rac']=df1['routingAreaRef'].str.extract("(RoutingArea=\w+)",expand=True)
df1['rac']=df1['rac'].str.replace('RoutingArea=',"")

df1['uralist']= df1['uraRef'].str.extract("(Ura=\w+)",expand=True)
df1['uralist']=df1['uralist'].str.replace('Ura=',"")

df1= df1.copy()
df1['iubLink']= df1['iubLinkRef'].str.extract("(IubLink=\w+)",expand=True)
df1['iubLink']=df1['iubLink'].str[12:17]

df2= df1.dropna(thresh=100, axis='columns')

larea= pd.read_csv('C:/Dumps/LTE_total/W_LocationArea.txt', sep= '\t')

#larea['LocationAreaId']= larea['LocationAreaId'].astype(int) 

df3= pd.merge(df2,larea, left_on=['RNC_NAME', 'lac'], right_on= ['RNC_NAME','LocationAreaId'], how= 'left')

df3['lac']= df3['lac_y']

df4= df3.loc[:, ['RNC_NAME',
 'WBTS', 'iubLink','iubLinkRef',
 'WCELL_NAME',
 'lac',
 'rac',
 'uralist',
 'administrativeState',                 
 'antennaPosition_latitude',
 'antennaPosition_longitude',
 'availabilityStatus',
 'cId',
 'localCellId',
 'maximumTransmissionPower',
 'operationalState',
 'primaryCpichPower',
 'primaryScramblingCode',
 'qQualMin',
 'qRxLevMin',
 'uarfcnDl',
 'uarfcnUl']]

df4.drop_duplicates(subset=['RNC_NAME', 'WCELL_NAME'], inplace= True)

rncfunction= pd.read_csv('C:/Dumps/LTE_total/W_RncFunction.txt', sep='\t',usecols=['RNC_NAME', 'rncId',
                                                                         'mcc','mnc'])


df5= pd.merge(df4, rncfunction, left_on= ['RNC_NAME'], right_on=['RNC_NAME'], how='left') 


M3uAssociation= pd.read_csv('C:/Dumps/LTE_total/W_M3uAssociation.txt', sep='\t', usecols=['RNC_NAME', 'M3uAssociationId'])

df6= pd.merge(df5, M3uAssociation, left_on=['RNC_NAME'], right_on=['RNC_NAME'], how= 'left')
df6.drop_duplicates(subset=['RNC_NAME', 'WBTS', 'WCELL_NAME'], inplace=True)
#pd.set_option('display.max_columns',24)
df6['SIGLA']= df6['iubLink'].str[1:4]

arquivo_txt= 'C:/Camilo/' + 'W_UtranCell'+ '.txt'
# arquivo_csv= 'C:/Camilo/' + 'W_UtranCell'+ '_'+str(semana_atual)+'.csv'

df6.to_csv(arquivo_txt, sep='\t')
# df6.to_csv(arquivo_csv, encoding='ISO-8859-1', errors='ignore', index=None)

print('Finished')

# Novo MO GUtranFreqRelation
#### 1 - Criaçao da Coluna GUtranFrequency para obtermos o ssbFrequency, necessário para o join com o MO NRCellDU  

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\GUtranFreqRelation_*_part*.parquet")
# garante dtype string (evita erro com NaN)
s = df1['gUtranSyncSignalFrequencyRef'].astype('string')

# pega tanto vsDataGUtran... quanto GUtran...
pat = r'(?:(?:^|,)\s*(?:vsData)?GUtranSyncSignalFrequency=)(?P<freq>[^,]+)'

df1['GUtranSyncSignalFrequency'] = s.str.extract(pat)['freq']

# Now proceed with dropping the now redundant 'EUtranCellTDD' column
df1.rename(columns={'EUtranCell': 'EUtranCellFDD'}, inplace=True)

df1.columns.to_list()

# df1['GUtranFreqRelationId']= df1['GUtranFreqRelationId'].fillna('')+df1['gUtranFreqRelationId'].fillna('')

df1['GUtranFreqRelationId']= df1['gUtranFreqRelationId']

#parte= df1[df1['gUtranFreqRelationId'].isna()]

# df1.drop('gUtranFreqRelationId', axis=1,inplace=True)

# df1['GUtranFreqRelation']= df1['GUtranFreqRelationId'].str.replace('NR_', '')
#df1['GUtranFreqRelation']=df1['GUtranFreqRelation'].str.split('-', n=0)
df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df1= df1.dropna(thresh=1, axis='columns')

df1['GUtranFrequency'] = df1['GUtranSyncSignalFrequency'].str.extract(r'([^-\s]+)')
df1.drop_duplicates(subset=['eNB', 'EUtranCellFDD', 'GUtranFrequency'], inplace=True)

df1= df1[[
 'MO','gUtranSyncSignalFrequencyRef','UF','eNB','EUtranCellFDD','GUtranFrequency', 'GUtranSyncSignalFrequency','GUtranFreqRelationId','anrMeasOn',
 'b1ThrRsrpFreqOffset',
 'b1ThrRsrqFreqOffset',
 'cellReselectionPriority',
 'cellReselectionSubPriority',
 'connectedModeMobilityPrio',
 'deriveSsbIndexFromCell',
 'endcB1MeasPriority','pMaxNR',
 'qOffsetFreq',
 'qQualMin',
 'qRxLevMin',
 'threshXHigh',
 'threshXHighQ',
 'threshXLow',
 'threshXLowQ']]

arquivo_txt= 'C:/Camilo/' + 'GUtranFreqRelation'+ '_'+str(semana_atual)+'.txt'
# arquivo_csv= 'C:/Camilo/' + 'GUtranFreqRelation'+ '_'+str(semana_atual)+'.csv'

df1.to_csv(arquivo_txt, sep='\t', index=None)
# junta1.to_csv(arquivo_csv, encoding='ISO-8859-1', errors='ignore', index=None)

print('Ok')

## MO CapacityState

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\CapacityState_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df2= df1.dropna(thresh=100, axis='columns')

df2= df2[['MO','UF', 'eNB', 
 'capacityKey',
 'capacityStateId',
 'currentCapacityLimit_noLimit',
 'currentCapacityLimit_value',
 'description',
 'grantedCapacityLevel',
 'keyId',
 'licensedCapacityLimitReached',
 'licenseState',
 'serviceState']]

df2.to_csv('C:/Dumps/LTE_Total/CapacityState.txt', sep='\t', index=None)
print('Ok')